# 03 - Modern Function Calling (Current Best Practice)

This notebook demonstrates the **current** `tools` format for function calling, which replaced the deprecated `functions` format in November 2023.

## 🚀 Why Modern Format is Better
- ✅ **Multiple parallel function calls** in single request
- ✅ **Better performance** with reduced API round-trips
- ✅ **Cleaner response structure** with tool call IDs
- ✅ **Future-proof** extensibility for new tool types
- ✅ **Production-ready** with proper error handling

## 🎯 Learning Objectives
- Master the modern `tools` parameter syntax
- Handle multiple simultaneous function calls
- Implement proper error handling patterns
- Build production-ready function calling workflows

In [1]:
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize client
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT4O_API_VERSION")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version,
)

print("✅ Modern function calling setup complete!")
print("🚀 Ready to use the latest tools format!")

✅ Modern function calling setup complete!
🚀 Ready to use the latest tools format!


## 🛠️ Modern Tools Format

The current format uses:
- `tools` parameter (array of tool definitions)
- `tool_choice` parameter (to control calling behavior)
- `tool_calls` response (array of tool call results with unique IDs)

In [2]:
# Define tools using MODERN format
modern_tools = [
    {
        "type": "function",  # 🆕 MODERN: Explicit type specification
        "function": {        # 🆕 MODERN: Nested function definition
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

print("🛠️ Modern tool definition created:")
print(json.dumps(modern_tools[0], indent=2))

🛠️ Modern tool definition created:
{
  "type": "function",
  "function": {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "The city and state, e.g. San Francisco, CA"
        },
        "unit": {
          "type": "string",
          "enum": [
            "celsius",
            "fahrenheit"
          ],
          "description": "Temperature unit"
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


In [3]:
# Make a call using MODERN tools format
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Amsterdam today?"
    }
]

# Modern API call
modern_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    tools=modern_tools,      # 🆕 MODERN: Use tools parameter
    tool_choice="auto",     # 🆕 MODERN: Use tool_choice parameter
    max_tokens=800,
    temperature=0.7
)

print("📞 MODERN TOOL CALL RESULT:")
print("=" * 50)
message = modern_completion.choices[0].message
print(f"Role: {message.role}")
print(f"Content: {message.content}")
print(f"Tool Calls: {message.tool_calls}")
print("=" * 50)

📞 MODERN TOOL CALL RESULT:
Role: assistant
Content: None
Tool Calls: [ChatCompletionMessageToolCall(id='call_Ze2mK29axJAJYygHPAR0Ojdm', function=Function(arguments='{"location":"Amsterdam"}', name='get_current_weather'), type='function')]


## 🔍 Modern Response Analysis

The modern format provides much better structure and tracking:

In [4]:
# Analyze the modern response structure
if hasattr(message, 'tool_calls') and message.tool_calls:
    tool_calls = message.tool_calls
    print(f"🛠️ MODERN TOOL CALLS STRUCTURE ({len(tool_calls)} calls):")
    
    for i, tool_call in enumerate(tool_calls, 1):
        print(f"\n   📞 Tool Call #{i}:")
        print(f"      🆔 ID: {tool_call.id}")  # 🆕 MODERN: Unique tracking ID!
        print(f"      🎯 Type: {tool_call.type}")
        print(f"      📞 Function: {tool_call.function.name}")
        print(f"      📋 Arguments (raw): {tool_call.function.arguments}")
        
        # Parse the arguments
        try:
            args = json.loads(tool_call.function.arguments)
            print(f"      📋 Arguments (parsed): {args}")
            print(f"      📍 Location: {args.get('location')}")
            print(f"      🌡️ Unit: {args.get('unit', 'Not specified')}")
        except json.JSONDecodeError:
            print("      ❌ Failed to parse arguments as JSON")
    
    print(f"\n🏁 Finish Reason: {modern_completion.choices[0].finish_reason}")
    print("\n✨ Notice the unique tool_call_id for tracking!")
else:
    print("❌ No tool calls detected")

🛠️ MODERN TOOL CALLS STRUCTURE (1 calls):

   📞 Tool Call #1:
      🆔 ID: call_Ze2mK29axJAJYygHPAR0Ojdm
      🎯 Type: function
      📞 Function: get_current_weather
      📋 Arguments (raw): {"location":"Amsterdam"}
      📋 Arguments (parsed): {'location': 'Amsterdam'}
      📍 Location: Amsterdam
      🌡️ Unit: Not specified

🏁 Finish Reason: tool_calls

✨ Notice the unique tool_call_id for tracking!


## 🎯 Understanding tool_call_id

The `tool_call_id` is crucial for:
- **🔗 Tracking**: Links function results back to specific calls
- **🔄 Debugging**: Helps identify which function produced which result
- **📊 Parallel Processing**: Essential when multiple tools are called simultaneously

Let's see how this works in practice:

In [5]:
# Execute the function and track the ID
def get_current_weather(location, unit="celsius"):
    """Simulate a weather API call"""
    weather_data = {
        "location": location,
        "temperature": "18" if unit == "celsius" else "64",
        "unit": unit,
        "description": "Partly cloudy with light rain",
        "humidity": "78%",
        "wind_speed": "12 km/h",
        "timestamp": "2024-01-15 14:30:00"
    }
    return json.dumps(weather_data)

# Process each tool call with proper ID tracking
if hasattr(message, 'tool_calls') and message.tool_calls:
    # Add the assistant's tool call message
    messages.append({
        "role": "assistant",
        "content": message.content,
        "tool_calls": [  # 🆕 MODERN: Array of tool calls
            {
                "id": tool_call.id,
                "type": tool_call.type,
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            } for tool_call in message.tool_calls
        ]
    })
    
    # Execute each tool call and add results
    for tool_call in message.tool_calls:
        print(f"🔧 Executing tool call: {tool_call.id}")
        
        args = json.loads(tool_call.function.arguments)
        function_result = get_current_weather(
            location=args["location"],
            unit=args.get("unit", "celsius")
        )
        
        # Add tool result with ID tracking
        messages.append({
            "role": "tool",                    # 🆕 MODERN: "tool" role
            "tool_call_id": tool_call.id,     # 🆕 MODERN: Links back to specific call
            "content": function_result         # 🆕 MODERN: Tool result
        })
        
        print(f"   ✅ Result for {tool_call.id}: {len(function_result)} characters")
    
    print("\n🎯 All tool calls executed and linked with IDs!")
else:
    print("❌ No tool calls to execute")

🔧 Executing tool call: call_Ze2mK29axJAJYygHPAR0Ojdm
   ✅ Result for call_Ze2mK29axJAJYygHPAR0Ojdm: 193 characters

🎯 All tool calls executed and linked with IDs!


In [6]:
# Get the final response with tool results
final_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    tools=modern_tools,  # Still provide tool definitions
    max_tokens=800,
    temperature=0.7
)

print("🎯 FINAL AI RESPONSE (with weather data):")
print("=" * 60)
print(final_completion.choices[0].message.content)
print("=" * 60)

print("\n💰 TOKEN USAGE:")
print(f"   🥇 First call: {modern_completion.usage.total_tokens} tokens")
print(f"   🥈 Second call: {final_completion.usage.total_tokens} tokens")
print(f"   🔢 Total: {modern_completion.usage.total_tokens + final_completion.usage.total_tokens} tokens")

🎯 FINAL AI RESPONSE (with weather data):
The weather in Amsterdam today is partly cloudy with light rain. The temperature is 18°C, with a humidity of 78% and a wind speed of 12 km/h.

💰 TOKEN USAGE:
   🥇 First call: 99 tokens
   🥈 Second call: 207 tokens
   🔢 Total: 306 tokens


## 📊 Complete Conversation Analysis

Let's examine the full conversation structure:

In [7]:
print("📋 COMPLETE CONVERSATION STRUCTURE:")
print("=" * 50)

for i, msg in enumerate(messages, 1):
    print(f"\n📝 Message #{i}:")
    print(f"   👤 Role: {msg['role']}")
    
    if msg['role'] == 'user':
        print(f"   💬 Content: {msg['content'][:50]}...")
    
    elif msg['role'] == 'assistant':
        print(f"   💬 Content: {msg.get('content', 'None')}")
        if 'tool_calls' in msg:
            print(f"   🛠️ Tool Calls: {len(msg['tool_calls'])} calls")
            for tc in msg['tool_calls']:
                print(f"      📞 {tc['id']}: {tc['function']['name']}")
    
    elif msg['role'] == 'tool':
        print(f"   🆔 Tool Call ID: {msg['tool_call_id']}")
        print(f"   📊 Result: {len(msg['content'])} characters")

print("\n✨ Notice the clean, trackable structure!")

📋 COMPLETE CONVERSATION STRUCTURE:

📝 Message #1:
   👤 Role: user
   💬 Content: What's the weather like in Amsterdam today?...

📝 Message #2:
   👤 Role: assistant
   💬 Content: None
   🛠️ Tool Calls: 1 calls
      📞 call_Ze2mK29axJAJYygHPAR0Ojdm: get_current_weather

📝 Message #3:
   👤 Role: tool
   🆔 Tool Call ID: call_Ze2mK29axJAJYygHPAR0Ojdm
   📊 Result: 193 characters

✨ Notice the clean, trackable structure!


## ⚡ Modern Format Benefits Demonstrated

### ✅ What We've Achieved
- **🏷️ Unique ID Tracking**: Each tool call has a trackable ID
- **🔧 Clean Structure**: Organized tool calls and responses
- **📊 Better Debugging**: Clear linkage between calls and results
- **🚀 Future-Ready**: Built for multiple simultaneous calls

### 🔄 Workflow Comparison

**Legacy Flow:**
1. User asks → AI calls function → Get result → AI responds
2. Multiple rounds for multiple functions
3. Higher latency and token costs

**Modern Flow:**
1. User asks → AI calls tool(s) → Get result(s) → AI responds
2. Single round even for multiple tools
3. Lower latency and token costs

---

## 🎓 Key Takeaways

✅ **Modern Syntax**: Master the current `tools` format  
✅ **ID Tracking**: Understand tool_call_id importance  
✅ **Efficient Structure**: Cleaner conversation management  
✅ **Performance**: Better than legacy approach  

## 🚀 Next Steps

Ready to see the real power of modern function calling?

**Continue to `04_multiple_tool_calls.ipynb`** to learn how to call multiple tools simultaneously! 🎯

This is where the modern format truly shines! ⚡